In [54]:
import pandas as pd
import numpy as np
import matplotlib as mtp

In [55]:
df_gdp = pd.read_csv('arquivos_base/gdp.csv', decimal='.', thousands=',')

In [56]:
df_gdp

,Country,Region,Year,GDP_pp
0,Afghanistan,"Middle East, North Africa, and Greater Arabia",1/1/1901,613.99
1,Afghanistan,"Middle East, North Africa, and Greater Arabia",1/1/1906,624.04
2,Afghanistan,"Middle East, North Africa, and Greater Arabia",1/1/1911,634.25
3,Afghanistan,"Middle East, North Africa, and Greater Arabia",1/1/1916,647.28
4,Afghanistan,"Middle East, North Africa, and Greater Arabia",1/1/1921,662.40
...,...,...,...,...
4414,Zimbabwe,Sub-Saharan Africa,1/1/1991,782.09
4415,Zimbabwe,Sub-Saharan Africa,1/1/1996,781.50
4416,Zimbabwe,Sub-Saharan Africa,1/1/2001,719.96
4417,Zimbabwe,Sub-Saharan Africa,1/1/2006,520.17


In [57]:
df_gdp.info()

<class 'pandas.DataFrame'>
RangeIndex: 4419 entries, 0 to 4418
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Country   4419 non-null   str    
 1   Region    4419 non-null   str    
 2   Year      4419 non-null   str    
 3    GDP_pp   4419 non-null   float64
dtypes: float64(1), str(3)
memory usage: 138.2 KB


In [58]:
df_gdp.columns = 'Country', 'Region', 'Year', 'GDP_pp'
# erro ao chamar coluna 'GDP_pp'. Resolvido declarando colunas

In [59]:
df_gdp['Year'] = df_gdp['Year'].apply(lambda x: int(x.split('/')[2]))
# transformando coluna Year para INT, para facilitar operacao

# INFORME O PRIMEIRO VALOR REGISTRADO DE CADA PAIS

In [60]:
df_gdp['Year'].value_counts()
# nao ha uma confirmacao de aparicoes exatas dos paises por ano, oscila a quantidade de aparicoes conforme o ano.
# portanto na primeira amostragem ha risco de ter algum pais que inicie sua amostragem em um ano diferente dos demais.

Year
1991    193
1996    193
2001    193
1901    192
1906    192
1911    192
1916    192
1921    192
1926    192
1931    192
1936    192
1941    192
1946    192
1951    192
1956    192
1961    192
1966    192
1971    192
1976    192
1981    192
1986    192
2006    192
2011    192
Name: count, dtype: int64

In [61]:
df_gdp.groupby('Country').agg({'Year':'min'}).reset_index().sort_values(by='Year')
# um dos paises inicia a amostragem em 1991, diferente dos outros paises. Pais -> 'Kosovo'

,Country,Year
0,Afghanistan,1901
1,Albania,1901
2,Algeria,1901
3,Andorra,1901
4,Angola,1901
...,...,...
188,Venezuela,1901
189,Vietnam,1901
190,"Yemen, Rep.",1901
191,Zambia,1901


In [62]:
df_ano_minimo = df_gdp.groupby('Country').agg({'Year':'min', 'GDP_pp':'first'}).reset_index()

In [63]:
df_ano_minimo
# amostragem do primeiro valor registrado de cada pais

,Country,Year,GDP_pp
0,Afghanistan,1901,613.99
1,Albania,1901,1062.01
2,Algeria,1901,1807.76
3,Andorra,1901,3352.50
4,Angola,1901,525.76
...,...,...,...
188,Venezuela,1901,766.21
189,Vietnam,1901,572.96
190,"Yemen, Rep.",1901,729.39
191,Zambia,1901,532.38


# Informe as regioes com maior crescimento de PIB per capita no seculo passado

In [64]:
df_gdp['Year'].value_counts().sort_index()
# seculo passado ==  1996 <=

Year
1901    192
1906    192
1911    192
1916    192
1921    192
1926    192
1931    192
1936    192
1941    192
1946    192
1951    192
1956    192
1961    192
1966    192
1971    192
1976    192
1981    192
1986    192
1991    193
1996    193
2001    193
2006    192
2011    192
Name: count, dtype: int64

In [65]:
df_gdp_crescimento = df_gdp.groupby(['Year', 'Region']).agg({'GDP_pp':'mean'}).reset_index()

In [66]:
df_gdp_crescimento = df_gdp_crescimento.loc[(df_gdp_crescimento['Year'] == 1901) | (df_gdp_crescimento['Year'] == 1996)]
# selecionando dados de inicio e final (ate encerramento do seculo) para retornar a diferenca de valores

In [67]:
df_gdp_crescimento.rename(columns={'GDP_pp':'GDP_inicial'}, inplace=True)
# formatacao do DF

In [68]:
df_operacional = df_gdp_crescimento[df_gdp_crescimento['Year'] == 1996].reset_index()
# criando um DF de manipulacao dos dados

In [69]:
df_gdp_crescimento.drop(df_gdp_crescimento[df_gdp_crescimento['Year'] == 1996].index, axis=0, inplace=True)
df_gdp_crescimento.drop(columns='Year', inplace=True)
# formatacao

In [70]:
df_gdp_crescimento['GDP_final'] = df_operacional['GDP_inicial']
# alinhando valores pra executar diferenca de valores

In [71]:
df_gdp_crescimento['GDP_diferenca'] = ((df_gdp_crescimento['GDP_final'] / df_gdp_crescimento['GDP_inicial'])-1) * 100
# formulando a operacao de diferenca

In [72]:
df_gdp_crescimento.sort_values(by='GDP_diferenca').reset_index().drop(columns='index').tail(3)
# As 3 regioes com os maiores crescimentos de PIB no seculo passado (ordem crescente)

,Region,GDP_inicial,GDP_final,GDP_diferenca
5,Europe,2583.788478,17932.684894,594.046167
6,Asia,900.756296,7311.992963,711.761516
7,"Middle East, North Africa, and Greater Arabia",1164.350000,11145.343913,857.215950


# Preencha os anos ausentes em cada pais com uma estimativa, baseada na diferenca entre o proximo registro e o anterior

In [73]:
import numpy as np
df_gdp.sort_values(by=['Year', 'Country'], inplace=True)

In [74]:
anos_presentes = sorted(df_gdp['Year'].value_counts().index.tolist())

In [75]:
n = anos_presentes[0]
anos_ausentes = []
while True:
    if n not in anos_presentes:
        anos_ausentes.append(n)
    elif n == anos_presentes[-1]:
        break
    else:
        n+=1
    anos_ausentes = sorted(anos_ausentes)

KeyboardInterrupt: 

In [53]:
anos_presentes

[1901,
 1902,
 1903,
 1904,
 1905,
 1906,
 1907,
 1908,
 1909,
 1910,
 1911,
 1912,
 1913,
 1914,
 1915,
 1916,
 1917,
 1918,
 1919,
 1920,
 1921,
 1922,
 1923,
 1924,
 1925,
 1926,
 1927,
 1928,
 1929,
 1930,
 1931,
 1932,
 1933,
 1934,
 1935,
 1936,
 1937,
 1938,
 1939,
 1940,
 1941,
 1942,
 1943,
 1944,
 1945,
 1946,
 1947,
 1948,
 1949,
 1950,
 1951,
 1952,
 1953,
 1954,
 1955,
 1956,
 1957,
 1958,
 1959,
 1960,
 1961,
 1962,
 1963,
 1964,
 1965,
 1966,
 1967,
 1968,
 1969,
 1970,
 1971,
 1972,
 1973,
 1974,
 1975,
 1976,
 1977,
 1978,
 1979,
 1980,
 1981,
 1982,
 1983,
 1984,
 1985,
 1986,
 1987,
 1988,
 1989,
 1990,
 1991,
 1992,
 1993,
 1994,
 1995,
 1996,
 1997,
 1998,
 1999,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011]